<a href="https://www.kaggle.com/code/crystalcheung399/mirrorhorizon?scriptVersionId=334649303" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

# Cross-Model Mirror Horizon / VPE Notebook — Runtime Optimized

This Colab is designed for the next reviewer-facing experiment:

> Test whether mirror horizon / verified-mode coverage generalizes across model families and model scales, while keeping runtime practical.

It self-creates the rollout dataset from GSM8K and outputs:

```text
source, model, model_family, scale, protocol, task, problem_id, rollout_id,
verified, mode, weight, question, gold, pred_answer, generated
```

It addresses the main reviewer concerns:

1. **Cross-model validation** beyond one Qwen-only run.
2. **Runtime optimization** through global batched generation, left padding, SDPA, mixed precision, adaptive OOM backoff, and checkpoint/resume.
3. **Estimator clarity**: raw strict VPE is `-inf` at zero reachability; the finite reported scalar is smoothed VPE. All logs are natural logs.
4. **Coverage horizon without iid/exponential assumptions**: exact finite-population coverage from observed rollouts.
5. **Mode-map robustness**: heuristic, TF-IDF cluster, and hybrid modes.
6. **Bootstrap confidence intervals** over problems.
7. **Negative controls**: shuffled verifier labels, shuffled verified-mode labels, and random modes.
8. **Format diagnostics**: answer marker rate, numeric extraction rate, near-token-limit rate.
9. **Predictive check**: whether early-budget horizon predicts full-budget pass@B.

Recommended first real run:

```python
MODEL_NAMES = [
    "Qwen/Qwen2.5-0.5B-Instruct",
    "Qwen/Qwen2.5-1.5B-Instruct",
    "HuggingFaceTB/SmolLM2-1.7B-Instruct",
]
N_PROBLEMS = 50
N_ROLLOUTS = 32
MAX_NEW_TOKENS = 160
```

For a fast sanity check, set `FAST_DEBUG = True`.


## 1. Install and imports


In [ ]:
!pip -q install transformers accelerate datasets pandas numpy matplotlib scipy tqdm sentencepiece scikit-learn


In [ ]:
import os, re, math, json, time, random, gc, itertools, shutil
from pathlib import Path
from typing import Optional, List, Dict, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from scipy.stats import spearmanr, pearsonr
from math import comb

import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans

SEED = 13
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

device = "cuda" if torch.cuda.is_available() else "cpu"
OUT = Path("/content/mirror_horizon_cross_model_outputs")
OUT.mkdir(parents=True, exist_ok=True)

print("device:", device)
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))
print("torch:", torch.__version__)


## 2. Experiment config


In [ ]:
#@title Cross-model config

# Default: two Qwen scales + one non-Qwen family.
# Add 3B or more families after a first run completes.
MODEL_NAMES = [
    # "Qwen/Qwen2.5-0.5B-Instruct",
    # "Qwen/Qwen2.5-1.5B-Instruct",
    # "HuggingFaceTB/SmolLM2-1.7B-Instruct",
    "Qwen/Qwen2.5-3B-Instruct",
    "microsoft/Phi-3.5-mini-instruct",
    # "google/gemma-2-2b-it",  # may require HF access
]

TASK = "gsm8k"
N_PROBLEMS = 50       #@param {type:"integer"}
N_ROLLOUTS = 32       #@param {type:"integer"}
MAX_NEW_TOKENS = 160  #@param {type:"integer"}
TEMPERATURE = 0.8     #@param {type:"number"}
TOP_P = 0.95          #@param {type:"number"}

PROMPT_STYLE = "strict_cot" #@param ["strict_cot", "direct_answer", "concise_cot"]
USE_CHAT_TEMPLATE = True #@param {type:"boolean"}

DEFAULT_BATCH_SIZE = 8 #@param {type:"integer"}
FAST_DEBUG = False #@param {type:"boolean"}

# Mode maps for analysis. Generation is shared; changing this list does not require GPU rerun.
ANALYSIS_MODE_MAPS = ["heuristic", "tfidf", "hybrid"]
MAX_TFIDF_CLUSTERS = 8 #@param {type:"integer"}

# Exact finite-population coverage budgets.
B_VALUES = [1, 2, 4, 8, 16, 32]

# Statistical settings.
N_BOOTSTRAP = 5000 #@param {type:"integer"}

# Generation precision.
DTYPE_MODE = "auto" #@param ["auto", "float16", "bfloat16", "float32"]
RESUME = True #@param {type:"boolean"}

# Optional: only use first model and a tiny subset for testing.
if FAST_DEBUG:
    MODEL_NAMES = MODEL_NAMES[:1]
    N_PROBLEMS = min(N_PROBLEMS, 8)
    N_ROLLOUTS = min(N_ROLLOUTS, 8)
    MAX_NEW_TOKENS = min(MAX_NEW_TOKENS, 80)
    DEFAULT_BATCH_SIZE = min(DEFAULT_BATCH_SIZE, 8)
    N_BOOTSTRAP = 500

def choose_dtype():
    if device != "cuda":
        return torch.float32
    if DTYPE_MODE == "float16":
        return torch.float16
    if DTYPE_MODE == "bfloat16":
        return torch.bfloat16
    if DTYPE_MODE == "float32":
        return torch.float32
    major, minor = torch.cuda.get_device_capability(0)
    return torch.bfloat16 if major >= 8 else torch.float16

TORCH_DTYPE = choose_dtype()

PROTOCOL = f"{TASK}_{PROMPT_STYLE}_tok{MAX_NEW_TOKENS}_temp{TEMPERATURE}_topp{TOP_P}_roll{N_ROLLOUTS}"

print(json.dumps({
    "models": MODEL_NAMES,
    "protocol": PROTOCOL,
    "n_problems": N_PROBLEMS,
    "n_rollouts": N_ROLLOUTS,
    "max_new_tokens": MAX_NEW_TOKENS,
    "temperature": TEMPERATURE,
    "top_p": TOP_P,
    "prompt_style": PROMPT_STYLE,
    "default_batch_size": DEFAULT_BATCH_SIZE,
    "analysis_mode_maps": ANALYSIS_MODE_MAPS,
    "dtype": str(TORCH_DTYPE),
    "device": device,
}, indent=2))


## 3. Dataset, verifier, and mode helpers


In [ ]:
def normalize_number_str(x: str) -> Optional[str]:
    if x is None:
        return None
    x = str(x).replace(",", "").strip()
    m = re.search(r"-?\d+(?:\.\d+)?", x)
    if not m:
        return None
    num = m.group(0)
    try:
        val = float(num)
        if abs(val - round(val)) < 1e-9:
            return str(int(round(val)))
        return str(val)
    except Exception:
        return num

def extract_gsm8k_gold(answer: str) -> Optional[str]:
    if "####" in answer:
        return normalize_number_str(answer.split("####")[-1])
    return normalize_number_str(answer)

def extract_model_answer(text: str) -> Optional[str]:
    if text is None:
        return None
    patterns = [
        r"####\s*(-?\d+(?:,\d{3})*(?:\.\d+)?)",
        r"[Tt]he answer is\s*(-?\d+(?:,\d{3})*(?:\.\d+)?)",
        r"[Aa]nswer:\s*(-?\d+(?:,\d{3})*(?:\.\d+)?)",
        r"[Ff]inal answer:\s*(-?\d+(?:,\d{3})*(?:\.\d+)?)",
        r"=+\s*(-?\d+(?:,\d{3})*(?:\.\d+)?)\s*$",
    ]
    for pat in patterns:
        m = re.search(pat, text.strip())
        if m:
            return normalize_number_str(m.group(1))
    nums = re.findall(r"-?\d+(?:,\d{3})*(?:\.\d+)?", text.replace(",", ""))
    if nums:
        return normalize_number_str(nums[-1])
    return None

def has_answer_marker(text: str) -> bool:
    if text is None:
        return False
    return bool(re.search(r"(Answer:|Final answer:|The answer is|####)", text, re.I))

def verify_numeric(generated_text: str, gold: str) -> int:
    pred = extract_model_answer(generated_text)
    return int(pred is not None and gold is not None and pred == gold)

def heuristic_solution_mode(text: str, verified: int) -> str:
    if not verified:
        return "invalid"

    words = text.split()
    if len(words) < 40:
        length_bin = "short"
    elif len(words) < 100:
        length_bin = "medium"
    else:
        length_bin = "long"

    ops = []
    if "+" in text or re.search(r"\bplus\b|\badd", text, re.I): ops.append("add")
    if "-" in text or re.search(r"\bminus\b|\bsubtract|\bleft|remain", text, re.I): ops.append("sub")
    if "*" in text or "×" in text or re.search(r"\btimes\b|\bmultiply|each|total", text, re.I): ops.append("mul")
    if "/" in text or "÷" in text or re.search(r"\bdivide|\bper\b|average", text, re.I): ops.append("div")
    if not ops:
        ops = ["none"]

    eq_count = len(re.findall(r"=", text))
    if eq_count == 0:
        eq_bin = "eq0"
    elif eq_count <= 2:
        eq_bin = "eq1-2"
    else:
        eq_bin = "eq3+"

    has_steps = bool(re.search(r"step|first|then|next|finally|therefore", text, re.I))
    style = "step" if has_steps else "direct"

    return f"{length_bin}|{'+'.join(sorted(set(ops)))}|{eq_bin}|{style}"

def approx_param_count_from_name(name: str) -> float:
    m = re.search(r"(\d+(?:\.\d+)?)B", name)
    if m:
        return float(m.group(1)) * 1e9
    m = re.search(r"(\d+(?:\.\d+)?)M", name)
    if m:
        return float(m.group(1)) * 1e6
    return np.nan

def model_family(name: str) -> str:
    low = name.lower()
    if "qwen" in low:
        return "Qwen"
    if "smollm" in low:
        return "SmolLM"
    if "phi" in low:
        return "Phi"
    if "gemma" in low:
        return "Gemma"
    if "llama" in low:
        return "Llama"
    if "mistral" in low:
        return "Mistral"
    return name.split("/")[0]

# Robust GSM8K loading.
try:
    gsm = load_dataset("openai/gsm8k", "main", split="test")
    GSM8K_DATASET_ID = "openai/gsm8k"
except Exception as e_openai:
    print("openai/gsm8k failed:", repr(e_openai))
    gsm = load_dataset("gsm8k", "main", split="test")
    GSM8K_DATASET_ID = "gsm8k"

gsm = gsm.shuffle(seed=SEED).select(range(min(N_PROBLEMS, len(gsm))))
problems = []
for i, row in enumerate(gsm):
    gold = extract_gsm8k_gold(row["answer"])
    problems.append({
        "problem_id": f"gsm8k_{i:04d}",
        "question": row["question"],
        "gold": gold,
        "gold_full": row["answer"],
    })
problems_df = pd.DataFrame(problems)
print("Loaded:", GSM8K_DATASET_ID, "| problems:", len(problems_df))
display(problems_df.head())


## 4. Fast model loading and batched generation


In [ ]:
def batch_size_for_model(name: str) -> int:
    # conservative caps to avoid OOM in Colab
    if any(x in name for x in ["7B", "8B"]):
        return min(DEFAULT_BATCH_SIZE, 1)
    if any(x in name for x in ["3B", "4B", "mini"]):
        return min(DEFAULT_BATCH_SIZE, 4)
    if "1.5B" in name or "1.7B" in name or "2B" in name:
        return min(DEFAULT_BATCH_SIZE, 8)
    return DEFAULT_BATCH_SIZE

def load_model(model_name: str):
    tok = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    tok.padding_side = "left"
    tok.truncation_side = "left"

    kwargs = dict(
        torch_dtype=TORCH_DTYPE,
        trust_remote_code=True,
        device_map="auto" if device == "cuda" else None,
    )
    try:
        model = AutoModelForCausalLM.from_pretrained(
            model_name,
            attn_implementation="sdpa",
            **kwargs,
        )
    except TypeError:
        model = AutoModelForCausalLM.from_pretrained(model_name, **kwargs)

    if device != "cuda":
        model.to(device)
    model.eval()
    model.config.use_cache = True
    return model, tok

def model_input_device(model):
    return next(model.parameters()).device

def base_user_prompt(question: str) -> str:
    if PROMPT_STYLE == "direct_answer":
        return (
            "Solve the following grade-school math problem. "
            "Return only the final number in the format 'Answer: <number>'.\n\n"
            f"Problem: {question}"
        )
    if PROMPT_STYLE == "concise_cot":
        return (
            "Solve the following grade-school math problem. "
            "Use concise reasoning. End with exactly 'Answer: <number>'.\n\n"
            f"Problem: {question}"
        )
    # strict_cot default
    return (
        "Solve the following grade-school math problem. "
        "Show your reasoning briefly, and end with a line exactly like 'Answer: <number>'.\n\n"
        f"Problem: {question}"
    )

def format_prompt(tokenizer, question: str) -> str:
    user = base_user_prompt(question)
    if USE_CHAT_TEMPLATE and hasattr(tokenizer, "apply_chat_template"):
        try:
            return tokenizer.apply_chat_template(
                [{"role": "user", "content": user}],
                tokenize=False,
                add_generation_prompt=True,
            )
        except Exception:
            return user
    return user

@torch.inference_mode()
def generate_batch(model, tokenizer, prompts: list[str], max_new_tokens: int) -> list[str]:
    inputs = tokenizer(
        prompts,
        return_tensors="pt",
        padding=True,
        truncation=True,
    ).to(model_input_device(model))
    input_len = inputs["input_ids"].shape[1]
    gen = model.generate(
        **inputs,
        do_sample=True,
        temperature=TEMPERATURE,
        top_p=TOP_P,
        max_new_tokens=max_new_tokens,
        use_cache=True,
        pad_token_id=tokenizer.eos_token_id,
    )
    return [tokenizer.decode(gen[i][input_len:], skip_special_tokens=True).strip() for i in range(len(prompts))]

def generate_batch_with_oom_backoff(model, tokenizer, prompts, init_batch_size, max_new_tokens):
    outputs = []
    i = 0
    batch_size = max(1, int(init_batch_size))
    while i < len(prompts):
        current = prompts[i:i+batch_size]
        try:
            outs = generate_batch(model, tokenizer, current, max_new_tokens=max_new_tokens)
            outputs.extend(outs)
            i += batch_size
        except RuntimeError as e:
            if "out of memory" in str(e).lower() and batch_size > 1:
                print(f"CUDA OOM at batch_size={batch_size}; retrying with {batch_size//2}")
                torch.cuda.empty_cache()
                gc.collect()
                batch_size = max(1, batch_size // 2)
            else:
                raise
    return outputs


## 5. Generate / resume cross-model rollouts


In [ ]:
rollout_path = OUT / f"verified_rollouts_crossmodel_{PROTOCOL}.csv"

if RESUME and rollout_path.exists():
    rollout_df = pd.read_csv(rollout_path)
    print("Resuming from:", rollout_path, "rows:", len(rollout_df))
else:
    rollout_df = pd.DataFrame()

existing_keys = set()
if len(rollout_df):
    existing_keys = set(zip(rollout_df["model"], rollout_df["protocol"], rollout_df["problem_id"], rollout_df["rollout_id"]))

runtime_log = []

for model_name in MODEL_NAMES:
    print("\n=== Loading model:", model_name, "===")
    t_model_start = time.time()
    model, tokenizer = load_model(model_name)
    scale = approx_param_count_from_name(model_name)
    family = model_family(model_name)
    init_batch_size = batch_size_for_model(model_name)
    print("family:", family, "| scale:", scale, "| initial batch size:", init_batch_size)

    tasks = []
    for prob in problems:
        prompt = format_prompt(tokenizer, prob["question"])
        # token length for length-sorted batching
        prompt_len = len(tokenizer(prompt, add_special_tokens=False).input_ids)
        for r in range(N_ROLLOUTS):
            rollout_id = f"{prob['problem_id']}_r{r:03d}"
            key = (model_name, PROTOCOL, prob["problem_id"], rollout_id)
            if key not in existing_keys:
                tasks.append({
                    "model": model_name,
                    "model_family": family,
                    "scale": scale,
                    "protocol": PROTOCOL,
                    "task": TASK,
                    "problem_id": prob["problem_id"],
                    "rollout_id": rollout_id,
                    "prompt": prompt,
                    "prompt_len": prompt_len,
                    "question": prob["question"],
                    "gold": prob["gold"],
                })

    # Sort by prompt length to reduce padding overhead.
    tasks = sorted(tasks, key=lambda x: x["prompt_len"])
    print("missing rollouts:", len(tasks))
    if not tasks:
        del model
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        gc.collect()
        continue

    new_rows = []
    batch_size = init_batch_size
    for start in tqdm(range(0, len(tasks), batch_size), desc=f"batched rollouts {model_name}"):
        chunk = tasks[start:start+batch_size]
        prompts = [x["prompt"] for x in chunk]
        t0 = time.time()
        completions = generate_batch_with_oom_backoff(model, tokenizer, prompts, batch_size, MAX_NEW_TOKENS)
        dt = time.time() - t0

        for task_row, gen_text in zip(chunk, completions):
            verified = verify_numeric(gen_text, task_row["gold"])
            pred = extract_model_answer(gen_text)
            heuristic_mode = heuristic_solution_mode(gen_text, verified)
            gen_tok_count = len(tokenizer(gen_text, add_special_tokens=False).input_ids)
            row = {
                "source": "real_gsm8k_rollout",
                "model": task_row["model"],
                "model_family": task_row["model_family"],
                "scale": task_row["scale"],
                "protocol": task_row["protocol"],
                "task": task_row["task"],
                "problem_id": task_row["problem_id"],
                "rollout_id": task_row["rollout_id"],
                "verified": verified,
                "heuristic_mode": heuristic_mode,
                "mode": heuristic_mode,
                "weight": 1.0,
                "prompt_style": PROMPT_STYLE,
                "temperature": TEMPERATURE,
                "top_p": TOP_P,
                "max_new_tokens": MAX_NEW_TOKENS,
                "question": task_row["question"],
                "gold": task_row["gold"],
                "pred_answer": pred,
                "answer_marker": int(has_answer_marker(gen_text)),
                "pred_answer_present": int(pred is not None),
                "generated_token_count": gen_tok_count,
                "near_max_token": int(gen_tok_count >= int(0.90 * MAX_NEW_TOKENS)),
                "generated": gen_text,
            }
            new_rows.append(row)

        runtime_log.append({
            "model": model_name,
            "model_family": family,
            "batch_prompts": len(chunk),
            "seconds": dt,
            "sec_per_completion": dt / max(len(chunk), 1),
        })

        # Checkpoint after every batch.
        tmp = pd.concat([rollout_df, pd.DataFrame(new_rows)], ignore_index=True)
        tmp = tmp.drop_duplicates(subset=["model","protocol","problem_id","rollout_id"], keep="last")
        tmp.to_csv(rollout_path, index=False)

    rollout_df = pd.concat([rollout_df, pd.DataFrame(new_rows)], ignore_index=True)
    rollout_df = rollout_df.drop_duplicates(subset=["model","protocol","problem_id","rollout_id"], keep="last")
    rollout_df.to_csv(rollout_path, index=False)

    print("model runtime minutes:", round((time.time() - t_model_start)/60, 2))
    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    gc.collect()

runtime_df = pd.DataFrame(runtime_log)
runtime_df.to_csv(OUT / f"runtime_log_{PROTOCOL}.csv", index=False)

print("Saved", rollout_path, "rows:", len(rollout_df))
display(rollout_df.head())
display(rollout_df.groupby(["model_family","model"])["verified"].agg(["mean","sum","count"]).round(4))
if len(runtime_df):
    display(runtime_df.groupby("model")["sec_per_completion"].agg(["mean","median","min","max","count"]).round(3))


## 6. Build alternative mode maps: heuristic, TF-IDF, hybrid


In [ ]:
def add_tfidf_modes(df: pd.DataFrame, max_clusters=8) -> pd.DataFrame:
    df = df.copy()
    df["tfidf_mode"] = "invalid"
    for (model, protocol, task), g in df.groupby(["model", "protocol", "task"]):
        idx = g.index[g["verified"] == 1].tolist()
        if len(idx) < 2:
            continue
        texts = df.loc[idx, "generated"].fillna("").astype(str).tolist()
        k = min(max_clusters, max(2, int(np.sqrt(len(texts)))), len(texts))
        try:
            vec = TfidfVectorizer(max_features=768, stop_words="english").fit_transform(texts)
            if vec.shape[0] < 2 or vec.shape[1] < 2:
                continue
            km = KMeans(n_clusters=k, random_state=SEED, n_init=10)
            labels = km.fit_predict(vec)
            for row_idx, lab in zip(idx, labels):
                df.at[row_idx, "tfidf_mode"] = f"tfidf_{lab}"
        except Exception as e:
            print("TF-IDF clustering failed for", model, protocol, task, ":", repr(e))
    return df

rollout_df = add_tfidf_modes(rollout_df, MAX_TFIDF_CLUSTERS)
rollout_df["hybrid_mode"] = np.where(
    rollout_df["verified"] == 1,
    rollout_df["heuristic_mode"].astype(str) + "||" + rollout_df["tfidf_mode"].astype(str),
    "invalid"
)

rollout_df.to_csv(rollout_path, index=False)
print("Mode maps created:", ["heuristic_mode", "tfidf_mode", "hybrid_mode"])
display(rollout_df[["model","problem_id","verified","heuristic_mode","tfidf_mode","hybrid_mode"]].head())


## 7. VPE metrics with estimator clarity


In [ ]:
EPS = 1e-12

def entropy_plugin_from_counts(counts: np.ndarray) -> float:
    counts = np.asarray(counts, dtype=float)
    total = counts.sum()
    if total <= 0:
        return 0.0
    p = counts[counts > 0] / total
    return float(-(p * np.log(p)).sum())

def entropy_miller_madow_from_counts(counts: np.ndarray) -> float:
    counts = np.asarray(counts, dtype=float)
    n = counts.sum()
    if n <= 0:
        return 0.0
    k = int((counts > 0).sum())
    h = entropy_plugin_from_counts(counts)
    # Natural-log Miller-Madow correction.
    return float(h + max(k - 1, 0) / (2 * n))

def mode_col_for(mode_map_name: str) -> str:
    return {
        "heuristic": "heuristic_mode",
        "tfidf": "tfidf_mode",
        "hybrid": "hybrid_mode",
    }[mode_map_name]

def compute_problem_vpe(df: pd.DataFrame, mode_map_name: str) -> pd.DataFrame:
    mode_col = mode_col_for(mode_map_name)
    rows = []
    for (model, family, scale, protocol, task, problem_id), g in df.groupby(["model","model_family","scale","protocol","task","problem_id"]):
        n = len(g)
        n_verified = int(g["verified"].sum())
        p = n_verified / n if n else 0.0
        p_smooth = (n_verified + 0.5) / (n + 1.0)
        verified_modes = g.loc[g["verified"] == 1, mode_col].astype(str)
        counts = verified_modes.value_counts().values
        h_plugin = entropy_plugin_from_counts(counts)
        h_mm = entropy_miller_madow_from_counts(counts)
        h_strict = (math.log(p) + h_plugin) if p > 0 else -np.inf
        h_eps = math.log(max(p, EPS)) + h_plugin
        h_smooth = math.log(p_smooth) + h_mm
        rows.append({
            "model": model,
            "model_family": family,
            "scale": scale,
            "protocol": protocol,
            "task": task,
            "problem_id": problem_id,
            "mode_map": mode_map_name,
            "n_rollouts": n,
            "n_verified": n_verified,
            "p_verified": p,
            "p_smooth": p_smooth,
            "pass_at_full": float(n_verified > 0),
            "zero_verified": float(n_verified == 0),
            "mode_entropy_plugin": h_plugin,
            "mode_entropy_mm": h_mm,
            "h_vpe_strict": h_strict,
            "h_vpe_eps": h_eps,
            "h_vpe_smooth": h_smooth,
            "n_verified_modes": int(len(set(verified_modes))),
            "mean_generated_tokens": float(g["generated_token_count"].mean()) if "generated_token_count" in g else np.nan,
            "answer_marker_rate": float(g["answer_marker"].mean()) if "answer_marker" in g else np.nan,
            "pred_answer_rate": float(g["pred_answer_present"].mean()) if "pred_answer_present" in g else np.nan,
            "near_max_token_rate": float(g["near_max_token"].mean()) if "near_max_token" in g else np.nan,
            "verified_given_pred_answer": float(g.loc[g["pred_answer_present"] == 1, "verified"].mean()) if "pred_answer_present" in g and (g["pred_answer_present"] == 1).any() else np.nan,
            "verified_given_answer_marker": float(g.loc[g["answer_marker"] == 1, "verified"].mean()) if "answer_marker" in g and (g["answer_marker"] == 1).any() else np.nan,
        })
    return pd.DataFrame(rows)

vpe_list = [compute_problem_vpe(rollout_df, m) for m in ANALYSIS_MODE_MAPS]
vpe_problem = pd.concat(vpe_list, ignore_index=True)
vpe_problem.to_csv(OUT / f"vpe_by_problem_crossmodel_{PROTOCOL}.csv", index=False)

summary_cols = [
    "p_verified","p_smooth","pass_at_full","zero_verified",
    "mode_entropy_plugin","mode_entropy_mm","h_vpe_eps","h_vpe_smooth",
    "n_verified","n_verified_modes","answer_marker_rate","pred_answer_rate",
    "near_max_token_rate","verified_given_pred_answer","verified_given_answer_marker"
]
vpe_summary = (
    vpe_problem.groupby(["mode_map","model_family","model","scale","protocol","task"])[summary_cols]
    .mean()
    .reset_index()
)
vpe_summary.to_csv(OUT / f"vpe_summary_crossmodel_{PROTOCOL}.csv", index=False)
display(vpe_summary.sort_values(["mode_map","scale"]).round(4))


## 8. Exact empirical coverage curves: no iid/exponential assumption


In [ ]:
def safe_comb(n: int, k: int) -> int:
    if k < 0 or k > n:
        return 0
    return comb(n, k)

def exact_finite_population_coverage_for_group(g: pd.DataFrame, mode_col: str, B_values: List[int]) -> List[dict]:
    N = len(g)
    verified_g = g[g["verified"] == 1]
    mode_counts = verified_g[mode_col].astype(str).value_counts().to_dict()
    n_verified = len(verified_g)
    rows = []
    denom_cache = {B: safe_comb(N, min(B, N)) for B in B_values}
    for B in B_values:
        b = min(B, N)
        denom = denom_cache[B]
        if denom <= 0:
            continue
        # exact pass@b under uniform subset from observed finite pool
        zero_prob = safe_comb(N - n_verified, b) / denom
        pass_prob = 1.0 - zero_prob
        K = 0.0
        for mode, c in mode_counts.items():
            miss_prob = safe_comb(N - c, b) / denom
            K += 1.0 - miss_prob
        H_cov = math.log1p(K)
        cond_modes = K / pass_prob if pass_prob > 0 else 0.0
        rows.append({
            "B": B,
            "K_expected_modes": K,
            "H_coverage": H_cov,
            "pass_at_B": pass_prob,
            "zero_coverage": zero_prob,
            "conditional_modes_given_success": cond_modes,
        })
    return rows

def compute_coverage(df: pd.DataFrame, mode_map_name: str) -> pd.DataFrame:
    mode_col = mode_col_for(mode_map_name)
    rows = []
    for (model, family, scale, protocol, task, problem_id), g in df.groupby(["model","model_family","scale","protocol","task","problem_id"]):
        for r in exact_finite_population_coverage_for_group(g, mode_col, B_VALUES):
            rows.append({
                "model": model,
                "model_family": family,
                "scale": scale,
                "protocol": protocol,
                "task": task,
                "problem_id": problem_id,
                "mode_map": mode_map_name,
                **r,
            })
    return pd.DataFrame(rows)

coverage_all = pd.concat([compute_coverage(rollout_df, m) for m in ANALYSIS_MODE_MAPS], ignore_index=True)
coverage_all.to_csv(OUT / f"coverage_by_problem_crossmodel_{PROTOCOL}.csv", index=False)

coverage_summary = (
    coverage_all.groupby(["mode_map","model_family","model","scale","protocol","task","B"])
    [["K_expected_modes","H_coverage","pass_at_B","zero_coverage","conditional_modes_given_success"]]
    .mean()
    .reset_index()
)
coverage_summary.to_csv(OUT / f"coverage_summary_crossmodel_{PROTOCOL}.csv", index=False)
display(coverage_summary.query("mode_map == @ANALYSIS_MODE_MAPS[0]").head(20).round(4))


## 9. Bootstrap confidence intervals over problems


In [ ]:
def bootstrap_ci_over_problems(df: pd.DataFrame, group_cols: List[str], metric_cols: List[str], n_boot=N_BOOTSTRAP, seed=SEED) -> pd.DataFrame:
    rng = np.random.default_rng(seed)
    out_rows = []
    for keys, g in df.groupby(group_cols):
        if not isinstance(keys, tuple):
            keys = (keys,)
        key_dict = dict(zip(group_cols, keys))
        problems = sorted(g["problem_id"].unique())
        m = len(problems)
        problem_to_values = g.groupby("problem_id")[metric_cols].mean()
        estimates = problem_to_values.mean()
        boots = {c: [] for c in metric_cols}
        for _ in range(n_boot):
            sample_probs = rng.choice(problems, size=m, replace=True)
            sample_vals = problem_to_values.loc[sample_probs]
            means = sample_vals.mean()
            for c in metric_cols:
                boots[c].append(means[c])
        for c in metric_cols:
            arr = np.array(boots[c], dtype=float)
            out_rows.append({
                **key_dict,
                "metric": c,
                "estimate": float(estimates[c]),
                "ci_low": float(np.nanquantile(arr, 0.025)),
                "ci_high": float(np.nanquantile(arr, 0.975)),
                "n_problems": m,
                "n_bootstrap": n_boot,
            })
    return pd.DataFrame(out_rows)

vpe_metric_cols = [
    "p_verified","p_smooth","pass_at_full","zero_verified",
    "mode_entropy_plugin","mode_entropy_mm","h_vpe_eps","h_vpe_smooth",
    "n_verified_modes","answer_marker_rate","pred_answer_rate","near_max_token_rate"
]
vpe_boot = bootstrap_ci_over_problems(
    vpe_problem,
    ["mode_map","model_family","model","scale","protocol","task"],
    vpe_metric_cols,
)
vpe_boot.to_csv(OUT / f"vpe_metric_bootstrap_ci_crossmodel_{PROTOCOL}.csv", index=False)

coverage_metric_cols = ["K_expected_modes","H_coverage","pass_at_B","zero_coverage","conditional_modes_given_success"]
coverage_boot = bootstrap_ci_over_problems(
    coverage_all,
    ["mode_map","model_family","model","scale","protocol","task","B"],
    coverage_metric_cols,
)
coverage_boot.to_csv(OUT / f"coverage_bootstrap_ci_crossmodel_{PROTOCOL}.csv", index=False)

display(vpe_boot.head().round(4))
display(coverage_boot.head().round(4))


## 10. Paired model comparisons at full rollout budget


In [ ]:
def paired_model_comparisons_at_B(coverage_df: pd.DataFrame, B_full: int = None, metric: str = "H_coverage", n_boot=N_BOOTSTRAP):
    if B_full is None:
        B_full = max(B_VALUES)
    rng = np.random.default_rng(SEED + 1)
    rows = []
    for (mode_map, protocol, task), gg in coverage_df.query("B == @B_full").groupby(["mode_map","protocol","task"]):
        models = sorted(gg["model"].unique())
        for a, b in itertools.combinations(models, 2):
            pa = gg[gg["model"] == a][["problem_id", metric]].rename(columns={metric: "a"})
            pb = gg[gg["model"] == b][["problem_id", metric]].rename(columns={metric: "b"})
            merged = pa.merge(pb, on="problem_id", how="inner")
            if len(merged) < 2:
                continue
            diffs = (merged["a"] - merged["b"]).values
            boot = []
            for _ in range(n_boot):
                boot.append(rng.choice(diffs, size=len(diffs), replace=True).mean())
            boot = np.array(boot)
            p = 2 * min((boot <= 0).mean(), (boot >= 0).mean())
            rows.append({
                "mode_map": mode_map,
                "protocol": protocol,
                "task": task,
                "B": B_full,
                "metric": metric,
                "model_a": a,
                "model_b": b,
                "estimate_a_minus_b": float(diffs.mean()),
                "ci_low": float(np.quantile(boot, 0.025)),
                "ci_high": float(np.quantile(boot, 0.975)),
                "p_two_sided": float(max(p, 1/(n_boot+1))),
                "n_problems": len(diffs),
            })
    return pd.DataFrame(rows)

paired_Bfull = pd.concat([
    paired_model_comparisons_at_B(coverage_all, max(B_VALUES), "H_coverage"),
    paired_model_comparisons_at_B(coverage_all, max(B_VALUES), "K_expected_modes"),
    paired_model_comparisons_at_B(coverage_all, max(B_VALUES), "pass_at_B"),
], ignore_index=True)
paired_Bfull.to_csv(OUT / f"paired_model_comparisons_B{max(B_VALUES)}_{PROTOCOL}.csv", index=False)
display(paired_Bfull.round(4))


## 11. Negative controls


In [ ]:
def make_negative_control_df(df: pd.DataFrame, control: str, mode_col: str = "heuristic_mode") -> pd.DataFrame:
    rng = np.random.default_rng(SEED + 7)
    out = df.copy()
    out["control"] = control
    out["analysis_mode"] = out[mode_col].astype(str)

    for (model, protocol, task), idx in out.groupby(["model","protocol","task"]).groups.items():
        idx = list(idx)
        if control == "shuffle_verified":
            vals = out.loc[idx, "verified"].values.copy()
            rng.shuffle(vals)
            out.loc[idx, "verified"] = vals
            out.loc[idx, "analysis_mode"] = np.where(out.loc[idx, "verified"].values == 1, out.loc[idx, "analysis_mode"], "invalid")
        elif control == "shuffle_verified_modes":
            v_idx = [i for i in idx if out.at[i, "verified"] == 1]
            modes = out.loc[v_idx, "analysis_mode"].values.copy()
            if len(modes) > 1:
                rng.shuffle(modes)
                out.loc[v_idx, "analysis_mode"] = modes
        elif control == "random_verified_modes":
            v_idx = [i for i in idx if out.at[i, "verified"] == 1]
            observed_modes = [m for m in out.loc[v_idx, "analysis_mode"].astype(str).unique() if m != "invalid"]
            if len(observed_modes) > 0:
                out.loc[v_idx, "analysis_mode"] = rng.choice(observed_modes, size=len(v_idx), replace=True)
        else:
            raise ValueError(control)
    return out

def compute_coverage_from_arbitrary_mode_col(df: pd.DataFrame, mode_col: str, label: str) -> pd.DataFrame:
    rows = []
    for (model, family, scale, protocol, task, problem_id), g in df.groupby(["model","model_family","scale","protocol","task","problem_id"]):
        for r in exact_finite_population_coverage_for_group(g, mode_col, B_VALUES):
            rows.append({
                "control": label,
                "model": model,
                "model_family": family,
                "scale": scale,
                "protocol": protocol,
                "task": task,
                "problem_id": problem_id,
                "B": r["B"],
                "K_expected_modes": r["K_expected_modes"],
                "H_coverage": r["H_coverage"],
                "pass_at_B": r["pass_at_B"],
                "zero_coverage": r["zero_coverage"],
            })
    return pd.DataFrame(rows)

control_frames = []
# Real baseline uses heuristic mode for a simple reviewer-facing negative-control comparison.
real_for_control = rollout_df.copy()
real_for_control["analysis_mode"] = real_for_control["heuristic_mode"].astype(str)
control_frames.append(compute_coverage_from_arbitrary_mode_col(real_for_control, "analysis_mode", "real"))

for control in ["shuffle_verified", "shuffle_verified_modes", "random_verified_modes"]:
    ctl = make_negative_control_df(rollout_df, control, "heuristic_mode")
    control_frames.append(compute_coverage_from_arbitrary_mode_col(ctl, "analysis_mode", control))

negative_cov = pd.concat(control_frames, ignore_index=True)
negative_summary = (
    negative_cov.groupby(["control","model_family","model","scale","protocol","task","B"])
    [["K_expected_modes","H_coverage","pass_at_B","zero_coverage"]]
    .mean()
    .reset_index()
)
negative_cov.to_csv(OUT / f"negative_control_coverage_by_problem_{PROTOCOL}.csv", index=False)
negative_summary.to_csv(OUT / f"negative_control_coverage_summary_{PROTOCOL}.csv", index=False)
B_FULL = max(B_VALUES)
display(negative_summary[negative_summary["B"].eq(B_FULL)].round(4))


## 12. Predictive checks: early horizon vs full-budget success


In [ ]:
def predictive_checks(coverage_df: pd.DataFrame, early_Bs=(4,8,16), full_B=None):
    if full_B is None:
        full_B = max(B_VALUES)
    rows = []
    full = coverage_df[coverage_df["B"] == full_B][["mode_map","model","protocol","task","problem_id","pass_at_B","K_expected_modes","H_coverage"]].rename(
        columns={
            "pass_at_B": "pass_full",
            "K_expected_modes": "K_full",
            "H_coverage": "H_full",
        }
    )
    for B in early_Bs:
        early = coverage_df[coverage_df["B"] == B][["mode_map","model","protocol","task","problem_id","pass_at_B","K_expected_modes","H_coverage"]].rename(
            columns={
                "pass_at_B": "pass_early",
                "K_expected_modes": "K_early",
                "H_coverage": "H_early",
            }
        )
        merged = early.merge(full, on=["mode_map","model","protocol","task","problem_id"], how="inner")
        for (mode_map, model, protocol, task), g in merged.groupby(["mode_map","model","protocol","task"]):
            for pred in ["pass_early", "K_early", "H_early"]:
                for target in ["pass_full", "K_full", "H_full"]:
                    sub = g[[pred, target]].dropna()
                    if len(sub) >= 5 and sub[pred].std() > 0 and sub[target].std() > 0:
                        rho, p = spearmanr(sub[pred], sub[target])
                        rows.append({
                            "mode_map": mode_map,
                            "model": model,
                            "protocol": protocol,
                            "task": task,
                            "early_B": B,
                            "predictor": pred,
                            "target": target,
                            "spearman_rho": float(rho),
                            "p_value": float(p),
                            "n_problems": len(sub),
                        })
    return pd.DataFrame(rows)

pred_checks = predictive_checks(coverage_all)
pred_checks.to_csv(OUT / f"predictive_checks_crossmodel_{PROTOCOL}.csv", index=False)
display(pred_checks.sort_values(["mode_map","model","early_B","target"]).round(4).head(40))


## 13. Publication-ready plots


In [ ]:
FIG = OUT / "figures"
FIG.mkdir(exist_ok=True)

def plot_metric_curve(boot_df: pd.DataFrame, mode_map="heuristic", metric="H_coverage", ylabel=None, title=None, filename=None):
    sub = boot_df[(boot_df["mode_map"] == mode_map) & (boot_df["metric"] == metric)].copy()
    if len(sub) == 0:
        print("No data for", mode_map, metric)
        return
    fig, ax = plt.subplots(figsize=(8,5))
    for model, g in sub.groupby("model"):
        g = g.sort_values("B")
        ax.plot(g["B"], g["estimate"], marker="o", label=model)
        ax.fill_between(g["B"], g["ci_low"], g["ci_high"], alpha=0.18)
    ax.set_xscale("log", base=2)
    ax.set_xlabel("rollout budget B")
    ax.set_ylabel(ylabel or metric)
    ax.set_title(title or f"{metric} ({mode_map})")
    ax.legend(fontsize=8)
    plt.tight_layout()
    if filename:
        plt.savefig(FIG / filename, dpi=220)
    plt.show()

plot_metric_curve(
    coverage_boot, "heuristic", "H_coverage",
    ylabel="mean log(1 + expected distinct verified modes)",
    title="Mirror horizon coverage curve with bootstrap CIs",
    filename="coverage_horizon_curve_ci.png",
)
plot_metric_curve(
    coverage_boot, "heuristic", "K_expected_modes",
    ylabel="expected distinct verified modes",
    title="Expected verified-mode coverage",
    filename="expected_modes_curve_ci.png",
)
plot_metric_curve(
    coverage_boot, "heuristic", "pass_at_B",
    ylabel="pass@B from observed rollout pool",
    title="Empirical pass@B curve",
    filename="pass_at_B_curve_ci.png",
)
plot_metric_curve(
    coverage_boot, "heuristic", "zero_coverage",
    ylabel="zero-coverage probability",
    title="Zero-coverage curve",
    filename="zero_coverage_curve_ci.png",
)

# VPE summary bar plot with CIs for smoothed VPE.
sub = vpe_boot[(vpe_boot["mode_map"] == "heuristic") & (vpe_boot["metric"] == "h_vpe_smooth")].copy()
if len(sub):
    sub = sub.sort_values("scale")
    fig, ax = plt.subplots(figsize=(8,4))
    x = np.arange(len(sub))
    ax.bar(x, sub["estimate"])
    ax.errorbar(x, sub["estimate"], yerr=[sub["estimate"]-sub["ci_low"], sub["ci_high"]-sub["estimate"]], fmt="none", capsize=4)
    ax.set_xticks(x)
    ax.set_xticklabels([m.replace("Qwen/Qwen2.5-","Qwen ").replace("-Instruct","") for m in sub["model"]], rotation=20, ha="right")
    ax.set_ylabel("smoothed VPE (nats)")
    ax.set_title("Smoothed VPE with bootstrap CIs")
    plt.tight_layout()
    plt.savefig(FIG / "smoothed_vpe_bar_ci.png", dpi=220)
    plt.show()

# Negative control plot at heuristic only.
sub = negative_summary.copy()
if len(sub):
    fig, ax = plt.subplots(figsize=(8,5))
    for (control, model), g in sub.groupby(["control","model"]):
        if control not in ["real","shuffle_verified","shuffle_verified_modes"]:
            continue
        g = g.sort_values("B")
        style = "-" if control == "real" else "--"
        alpha = 1.0 if control == "real" else 0.65
        ax.plot(g["B"], g["H_coverage"], style, marker="o", alpha=alpha, label=f"{model} | {control}")
    ax.set_xscale("log", base=2)
    ax.set_xlabel("rollout budget B")
    ax.set_ylabel("coverage horizon")
    ax.set_title("Negative controls for verified-mode coverage")
    ax.legend(fontsize=7, ncol=1)
    plt.tight_layout()
    plt.savefig(FIG / "negative_control_coverage_horizon.png", dpi=220)
    plt.show()

print("Figures saved to", FIG)


## 14. Save manifest and zip outputs


In [ ]:
manifest = {
    "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
    "models": MODEL_NAMES,
    "task": TASK,
    "n_problems": N_PROBLEMS,
    "n_rollouts": N_ROLLOUTS,
    "max_new_tokens": MAX_NEW_TOKENS,
    "temperature": TEMPERATURE,
    "top_p": TOP_P,
    "prompt_style": PROMPT_STYLE,
    "protocol": PROTOCOL,
    "analysis_mode_maps": ANALYSIS_MODE_MAPS,
    "B_values": B_VALUES,
    "n_bootstrap": N_BOOTSTRAP,
    "default_batch_size": DEFAULT_BATCH_SIZE,
    "dtype": str(TORCH_DTYPE),
    "gsm8k_dataset_id": GSM8K_DATASET_ID,
    "device": device,
}
with open(OUT / f"manifest_{PROTOCOL}.json", "w") as f:
    json.dump(manifest, f, indent=2)

print("Output files:")
for p in sorted(OUT.glob("*")):
    if p.is_file():
        print(p, p.stat().st_size)
print("Figure files:")
for p in sorted((OUT/"figures").glob("*")):
    print(p, p.stat().st_size)

zip_path = shutil.make_archive("/kaggle/working", "zip", str(OUT))
print("Zipped outputs:", zip_path)

# In Colab uncomment:
# from google.colab import files
# files.download(zip_path)


In [ ]:
# --- Kaggle download (replaces google.colab files.download) ---
from IPython.display import FileLink

# make sure the zip lives under /kaggle/working so Kaggle exposes it
import os, shutil
if not zip_path.startswith("/kaggle/working"):
    dst = os.path.join("/kaggle/working", os.path.basename(zip_path))
    shutil.copy(zip_path, dst)
    zip_path = dst

print("Saved to:", zip_path)
FileLink(zip_path)   # click this link to download, or use the Output tab on the right